# LLMs on ConTRoL — Google Colab Runner
Run NLI experiments on the ConTRoL dataset using Groq or HuggingFace models.

In [ ]:
# 1. Clone the repo and install dependencies
!git clone https://github.com/yuvalzohar12/computational_semantics_course.git
%cd computational_semantics_course
!pip install -q -r requirements.txt

In [ ]:
# 2. Mount Google Drive and copy the ConTRoL dataset
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
# Update this path to wherever ConTRoL-dataset lives in your Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/ConTRoL-dataset'

os.makedirs('data', exist_ok=True)
if os.path.exists(DRIVE_DATASET_PATH):
    shutil.copytree(DRIVE_DATASET_PATH, 'data/ConTRoL-dataset', dirs_exist_ok=True)
    print('Dataset copied successfully.')
else:
    print(f'Dataset not found at {DRIVE_DATASET_PATH} — update the path above.')

In [ ]:
# 3. Set API keys
import os

os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'   # replace with your key
os.environ['HF_API_KEY']   = ''   # only needed for qwen2.5-32b-instruct
os.environ['LAB_API_KEY']  = ''   # lab/Tailscale — not reachable from Colab

In [ ]:
# 4. Run an experiment
# Available experiments: zero_shot | few_shot_cot | hdqd_pipeline | q2_pipeline
# Available models (Colab-compatible): llama-3.1-8b-instant (Groq) | qwen2.5-32b-instruct (HF)

EXPERIMENT = 'zero_shot'
MODEL      = 'llama-3.1-8b-instant'
LIMIT      = 5   # set to None to run on the full dataset

limit_flag = f'--limit {LIMIT}' if LIMIT else ''
!python main.py --experiment {EXPERIMENT} --model {MODEL} {limit_flag}

In [ ]:
# 5. View results
import json, glob

result_files = sorted(glob.glob('results/*.json'))
if result_files:
    latest = result_files[-1]
    print(f'Latest result file: {latest}\n')
    with open(latest) as f:
        data = json.load(f)
    print('Metadata:', json.dumps(data['metadata'], indent=2))
    correct = sum(1 for s in data['samples'] if s.get('correct'))
    total   = len(data['samples'])
    print(f'\nAccuracy: {correct}/{total} = {correct/total:.1%}' if total else 'No samples yet.')
else:
    print('No result files found yet — run cell 4 first.')